# 01 — Pilot dataset audit

QC/coverage only; **no ML training**. Run merge, validation, and feature building first. Missing files are reported rather than replaced.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
PATH=Path('../data/processed/master_19papers_features.xlsx')
df=pd.read_excel(PATH)
df.shape

## Dimensions, provenance, row role, and independent groups

In [ ]:
display(df[['Paper_ID','DOI','Condition_ID','Experiment_Group_ID']].nunique())
display(df['Paper_ID'].value_counts(dropna=False))
role=next((c for c in ['Data_role','Data_Role','Row_Type'] if c in df), None)
if role: display(df[role].value_counts(dropna=False))
print('Independent identifiers:', df.Experiment_Group_ID.nunique())

## Missingness and composition coverage

In [ ]:
display((100*df.isna().mean()).sort_values(ascending=False).to_frame('missing_%'))
comp=[c for c in df if 'at%' in c.lower() or 'atomic_percent' in c.lower()]
display(df[comp].notna().sum().sort_values(ascending=False).to_frame('available_rows'))

## Temperature, strain rate, and SFE method coverage

In [ ]:
for term in ['temperature','strain_rate','sfe']:
 cols=[c for c in df if term in c.lower().replace(' ','_')]
 print(term, cols)
 if term=='sfe':
  for c in cols: display(df[c].value_counts(dropna=False).head(20))
for c in [c for c in df if 'temperature' in c.lower() or ('strain' in c.lower() and 'rate' in c.lower())]:
 pd.to_numeric(df[c],errors='coerce').hist(); plt.title(c); plt.show()

## Mechanism distributions (row and experiment-group level)

In [ ]:
for c in [x for x in ['TRIP','TWIP'] if x in df]:
 display(df[c].value_counts(dropna=False).to_frame('rows'))
 display(df.groupby('Experiment_Group_ID')[c].agg(lambda x: tuple(pd.unique(x.dropna()))).value_counts().to_frame('groups'))
if {'TRIP','TWIP'} <= set(df): display(df.groupby(['TRIP','TWIP'],dropna=False).size().to_frame('rows'))

## Figures

Generate versioned paths using `python -m src.analysis.dataset_audit`. Figures cover paper/label distributions and missingness; no estimator is fit.